# Induction and Abduction with PLN-THRML

Beyond chain inference (A→B→C), PLN supports **branching topologies**
where multiple implications share a common node.
This notebook demonstrates two fundamental patterns:

- **V-shape (induction)**: C→A and C→B — infer A→B.  
  "Two effects share a common cause → the effects are related."

- **Inverted-V (abduction)**: A→C and B→C — infer A→B.  
  "Two causes produce the same effect → the causes are related (explaining away)."

Both topologies are compiled to the same Boltzmann factor graph engine
used for chain inference, but the graph structure captures the different
independence assumptions.

---
## Setup

In [ ]:
%pip install -e ..

In [ ]:
from pln_thrml import (
    build_beta_v_graph, build_beta_inv_v_graph,
    sample_and_measure, run_beta_sampling, estimate_beta_marginal,
    DEFAULT_K, bin_centers,
)

---
## Induction (V-shape)

**Topology**: Root → Left, Root → Right (the Root "causes" both Left and Right).

```
    Root (C)
   /        \
  v          v
Left (A)   Right (B)
```

**Question**: Given that we know C→A and C→B, and A is observed (clamped),
what can the factor graph tell us about B?

**Example**: Rain causes wet streets (C→A, s=0.9, c=0.85) and rain causes
umbrellas (C→B, s=0.8, c=0.8). If we observe wet streets (A is true),
what's the posterior belief about umbrellas (B)?

In [ ]:
v_graph = build_beta_v_graph(
    root_prior=0.99, root_confidence=0.99,        # C: known condition (force True)
    left_strength=0.9, right_strength=0.8,         # C→A, C→B strengths
    left_impl_confidence=0.85, right_impl_confidence=0.8,
    left_background=0.02, right_background=0.02,
    left_prior=0.99, left_confidence=0.99,         # A: force condition=True
    right_prior=0.5, right_confidence=0.01,        # B: target (weak prior)
)

print(f"Graph nodes: {v_graph['n']}  (left=A, root=C, right=B)")
print(f"K = {v_graph['k']} bins per node")
print(f"Factors: {len(v_graph['factors'])}  (3 priors + 2 implications)")
print(f"Free blocks: {len(v_graph['free_blocks'])}")

In [ ]:
s_ind, c_ind = sample_and_measure(v_graph, v_graph["right"], seed=42)

print(f"Induction: P(B | A known, C→A, C→B)")
print(f"  strength  = {s_ind:.4f}")
print(f"  confidence = {c_ind:.4f}")

In [ ]:
import jax.numpy as jnp

samples_v = run_beta_sampling(v_graph, seed=42)
posterior_v, s_v, c_v = estimate_beta_marginal(samples_v, v_graph, v_graph["right"])

centers = bin_centers(DEFAULT_K)
print(f"Posterior histogram for B (K={DEFAULT_K} bins):")
print(f"  {posterior_v}")
print(f"\nPeak bin: {centers[jnp.argmax(posterior_v)]:.3f}")
print(f"Recovered: (stv {s_v:.4f} {c_v:.4f})")

### Interpretation

The factor graph propagates information along the V-shape:
A is clamped true → the Root→Left implication constrains Root (C) to be
likely true → the Root→Right implication then pushes Right (B) toward
the strength of C→B.

The inferred relationship between A and B emerges **indirectly** through
the shared cause C, which is exactly the PLN induction pattern.

---
## Abduction (Inverted-V)

**Topology**: Left → Center, Right → Center (both Left and Right "cause" Center).

```
Left (A)   Right (B)
  \          /
   v        v
   Center (C)
```

**Question**: Given that A→C and B→C, and A is observed (clamped true),
what can we infer about B? This is the classic **explaining away** pattern:
if A already explains C, is B still needed?

**Example**: Sprinkler causes wet grass (A→C, s=0.9, c=0.85) and
rain causes wet grass (B→C, s=0.8, c=0.8). If we know the sprinkler
was on (A is true), what do we believe about rain (B)?

In [ ]:
inv_v_graph = build_beta_inv_v_graph(
    left_prior=0.99, left_confidence=0.99,         # A: known condition
    right_prior=0.5, right_confidence=0.01,        # B: target (weak prior)
    left_strength=0.9, right_strength=0.8,         # A→C, B→C strengths
    left_impl_confidence=0.85, right_impl_confidence=0.8,
    left_background=0.02, right_background=0.02,
    center_prior=0.5, center_confidence=0.01,      # C: weak prior
)

print(f"Graph nodes: {inv_v_graph['n']}  (left=A, center=C, right=B)")
print(f"K = {inv_v_graph['k']} bins per node")
print(f"Factors: {len(inv_v_graph['factors'])}  (3 priors + 2 implications)")
print(f"Free blocks: {len(inv_v_graph['free_blocks'])}")

In [ ]:
s_abd, c_abd = sample_and_measure(inv_v_graph, inv_v_graph["right"], seed=42)

print(f"Abduction: P(B | A known, A→C, B→C)")
print(f"  strength  = {s_abd:.4f}")
print(f"  confidence = {c_abd:.4f}")

In [ ]:
samples_inv = run_beta_sampling(inv_v_graph, seed=42)
posterior_inv, s_inv, c_inv = estimate_beta_marginal(
    samples_inv, inv_v_graph, inv_v_graph["right"]
)

print(f"Posterior histogram for B (K={DEFAULT_K} bins):")
print(f"  {posterior_inv}")
print(f"\nPeak bin: {centers[jnp.argmax(posterior_inv)]:.3f}")
print(f"Recovered: (stv {s_inv:.4f} {c_inv:.4f})")

### Interpretation: Explaining Away

In the inverted-V topology, both A and B compete to explain C.
When A is clamped true, A already provides a strong explanation
for C via the A→C implication. This means B is **not required**
to explain C, so B's posterior stays closer to its weak prior.

Abduction typically has **higher error** than other PLN rules because
the factor graph must resolve competing causal explanations through
sampling, while PLN's closed-form formula uses an approximation
that doesn't fully capture this interaction.

---
## Comparison: Induction vs Abduction

In [ ]:
print(f"{'Topology':<25} {'Strength':>10} {'Confidence':>12}")
print(f"{'─' * 47}")
print(f"{'Induction (V-shape)':<25} {s_ind:>10.4f} {c_ind:>12.4f}")
print(f"{'Abduction (inverted-V)':<25} {s_abd:>10.4f} {c_abd:>12.4f}")
print()
print(f"Strength difference:  {abs(s_ind - s_abd):.4f}")
print(f"Confidence difference: {abs(c_ind - c_abd):.4f}")

### Key Observations

- **Induction** propagates through a shared cause: A ← C → B.  
  When A is clamped true, C is constrained, which in turn constrains B.
  Information flows cleanly through the common root.

- **Abduction** involves competing explanations: A → C ← B.  
  When A is clamped true, A already explains C, so B receives
  weaker indirect evidence. The "explaining away" effect makes
  the posterior for B less peaked.

- Abduction error (vs PLN closed-form) is typically higher because
  the factor graph captures the full joint distribution including
  inter-cause dependencies, while PLN's formula uses a simpler
  approximation.

---
## Next Steps

- **Run the full test suite** to see all 11 rules: `pytest tests/ -v`
- **Chain inference** walkthrough: see `01_quick_start.ipynb`